## Homework

> Note: sometimes your answer doesn't match one of the options exactly. 
> That's fine. 
> Select the option that's closest to your solution.


### Dataset

In this homework, we will use the Bank Marketing dataset. Download it from [here](https://archive.ics.uci.edu/static/public/222/bank+marketing.zip).

Or you can do it with `wget`:

```bash
wget https://archive.ics.uci.edu/static/public/222/bank+marketing.zip
```

We need to take `bank/bank-full.csv` file from the downloaded zip-file.  
In this dataset our desired target for classification task will be `y` variable - has the client subscribed a term deposit or not. 

### Features

For the rest of the homework, you'll need to use only these columns:

* `age`,
* `job`,
* `marital`,
* `education`,
* `balance`,
* `housing`,
* `contact`,
* `day`,
* `month`,
* `duration`,
* `campaign`,
* `pdays`,
* `previous`,
* `poutcome`,
* `y`

### Data preparation

* Select only the features from above.
* Check if the missing values are presented in the features.

### Question 1

What is the most frequent observation (mode) for the column `education`?

- `unknown`
- `primary`
- `secondary`
- `tertiary`

In [35]:
import pandas as pd
import numpy as np

In [36]:
df = pd.read_csv('../data/bank-full.csv', sep=";")
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [37]:
print(df.columns)

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')


In [38]:
# df.columns = df.columns.str.replace('"', '').str.strip()

In [67]:
df_selected = df[['age', 'job', 'marital', 'education', 'balance', 'housing', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y', 'loan']]

In [68]:
df_selected.isnull().sum()

age          0
job          0
marital      0
education    0
balance      0
housing      0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
loan         0
dtype: int64

In [78]:
df_selected.dtypes

age           int64
job          object
marital      object
education    object
balance       int64
housing      object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
loan         object
dtype: object

In [69]:
education_mode = df_selected['education'].mode()[0]
education_mode

'secondary'

### Question 2

Create the [correlation matrix](https://www.google.com/search?q=correlation+matrix) for the numerical features of your dataset. 
In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `age` and `balance`
- `day` and `campaign`
- `day` and `pdays`
- `pdays` and `previous`

In [70]:
numerical_features = df_selected[['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']]

correlation_matrix = numerical_features.corr()

print("Correlation Matrix:")
print(correlation_matrix)

correlation_pairs = correlation_matrix.unstack().sort_values(ascending=False)

strongest_correlation = correlation_pairs[correlation_pairs < 1].idxmax()

print(f"The two features with the biggest correlation are: {strongest_correlation}")


Correlation Matrix:
               age   balance       day  duration  campaign     pdays  previous
age       1.000000  0.097783 -0.009120 -0.004648  0.004760 -0.023758  0.001288
balance   0.097783  1.000000  0.004503  0.021560 -0.014578  0.003435  0.016674
day      -0.009120  0.004503  1.000000 -0.030206  0.162490 -0.093044 -0.051710
duration -0.004648  0.021560 -0.030206  1.000000 -0.084570 -0.001565  0.001203
campaign  0.004760 -0.014578  0.162490 -0.084570  1.000000 -0.088628 -0.032855
pdays    -0.023758  0.003435 -0.093044 -0.001565 -0.088628  1.000000  0.454820
previous  0.001288  0.016674 -0.051710  0.001203 -0.032855  0.454820  1.000000
The two features with the biggest correlation are: ('previous', 'pdays')


### Target encoding

* Now we want to encode the `y` variable.
* Let's replace the values `yes`/`no` with `1`/`0`.

### Split the data

* Split your data in train/val/test sets with 60%/20%/20% distribution.
* Use Scikit-Learn for that (the `train_test_split` function) and set the seed to `42`.
* Make sure that the target value `y` is not in your dataframe.

In [71]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score

In [61]:
df_selected['y'] = df_selected['y'].replace({'yes': 1, 'no': 0})

X = df_selected.drop(columns=['y'])  # Features (independent variables)
y = df_selected['y']                 # Target variable

# Split the data into train (60%), val (20%), and test (20%) sets
# First, split into train and temporary set (80/20)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)

# Then, split the temporary set into validation and test sets (50/50 of the remaining 40%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 27126
Validation set size: 9042
Test set size: 9043


/tmp/ipykernel_74533/2147534923.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_selected['y'] = df_selected['y'].replace({'yes': 1, 'no': 0})
/tmp/ipykernel_74533/2147534923.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['y'] = df_selected['y'].replace({'yes': 1, 'no': 0})


### Question 3

* Calculate the mutual information score between `y` and other categorical variables in the dataset. Use the training set only.
* Round the scores to 2 decimals using `round(score, 2)`.

Which of these variables has the biggest mutual information score?
  
- `contact`
- `education`
- `housing`
- `poutcome`

In [79]:
categorical_features = ['contact', 'education', 'housing', 'poutcome']

X_train_categorical = X_train[categorical_features].apply(lambda x: x.astype('category'))

mi_scores = mutual_info_classif(X_train_categorical, y_train, discrete_features=True)

mi_scores_df = pd.DataFrame({'Feature': categorical_features, 'Mutual Information Score': mi_scores})
mi_scores_df['Mutual Information Score'] = mi_scores_df['Mutual Information Score'].round(2)

print(mi_scores_df)

max_mi_feature = mi_scores_df.loc[mi_scores_df['Mutual Information Score'].idxmax()]
print(f"The variable with the biggest mutual information score is: {max_mi_feature['Feature']}")

     Feature  Mutual Information Score
0    contact                      0.01
1  education                      0.00
2    housing                      0.01
3   poutcome                      0.03
The variable with the biggest mutual information score is: poutcome


### Question 4

* Now let's train a logistic regression.
* Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
* Fit the model on the training dataset.
    - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
    - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
* Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.6
- 0.7
- 0.8
- 0.9

In [83]:
X.head()

,age,job,marital,education,balance,housing,contact,day,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,2143,yes,unknown,5,may,261,1,-1,0,unknown
1,44,technician,single,secondary,29,yes,unknown,5,may,151,1,-1,0,unknown
2,33,entrepreneur,married,secondary,2,yes,unknown,5,may,76,1,-1,0,unknown
3,47,blue-collar,married,unknown,1506,yes,unknown,5,may,92,1,-1,0,unknown
4,33,unknown,single,unknown,1,no,unknown,5,may,198,1,-1,0,unknown


In [87]:
categorical_features = ['job', 'marital', 'education', 'contact', 'housing', 'poutcome', 'month']

encoder = OneHotEncoder(drop='first', sparse_output=False)
X_encoded = encoder.fit_transform(X[categorical_features])

numerical_features = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
X_numerical = X[numerical_features].values

import numpy as np
X_prepared = np.hstack([X_numerical, X_encoded])

X_train, X_temp, y_train, y_temp = train_test_split(X_prepared, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_val_pred = model.predict(X_val)
accuracy_val = accuracy_score(y_val, y_val_pred)

accuracy_val_rounded = round(accuracy_val, 2)

print(f"Validation accuracy: {accuracy_val_rounded}")


Validation accuracy: 0.9


### Question 5 

* Let's find the least useful feature using the *feature elimination* technique.
* Train a model with all these features (using the same parameters as in Q4).
* Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
* For each feature, calculate the difference between the original accuracy and the accuracy without the feature. 

Which of following feature has the smallest difference?

- `age`
- `balance`
- `marital`
- `previous`

In [93]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_train_pred = model.predict(X_train)
baseline_accuracy = accuracy_score(y_train, y_train_pred)

tmp = df_selected[['age', 'balance', 'marital']]

feature_names = tmp.columns.tolist()  # List of feature names

accuracy_differences = {}

for feature in feature_names:

    feature_index = df_selected.columns.get_loc(feature)  # Get index of the feature
    X_train_reduced = np.delete(X_train, feature_index, axis=1)
    X_val_reduced = np.delete(X_val, feature_index, axis=1)

    model.fit(X_train_reduced, y_train)
    y_val_pred_reduced = model.predict(X_val_reduced)
    
    accuracy_reduced = accuracy_score(y_val, y_val_pred_reduced)

    accuracy_differences[feature] = baseline_accuracy - accuracy_reduced

least_useful_feature = min(accuracy_differences, key=accuracy_differences.get)
smallest_difference_value = accuracy_differences[least_useful_feature]

print(f"Baseline Accuracy: {baseline_accuracy:.2f}")
print(f"The least useful feature is: '{least_useful_feature}' with a difference of: {smallest_difference_value:.2f}")


Baseline Accuracy: 0.90
The least useful feature is: 'age' with a difference of: 0.00


### Question 6

* Now let's train a regularized logistic regression.
* Let's try the following values of the parameter `C`: `[0.01, 0.1, 1, 10, 100]`.
* Train models using all the features as in Q4.
* Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.01
- 0.1
- 1
- 10
- 100

> **Note**: If there are multiple options, select the smallest `C`.

In [94]:
C_values = [0.01, 0.1, 1, 10, 100]

accuracy_results = {}

for C in C_values:

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_val_pred = model.predict(X_val)
    
    accuracy_val = accuracy_score(y_val, y_val_pred)
    
    accuracy_results[C] = round(accuracy_val, 3)

best_C = min(accuracy_results, key=lambda k: (-accuracy_results[k], k))

print("Accuracy results for each C value:")
for C, accuracy in accuracy_results.items():
    print(f"C={C}: Accuracy={accuracy}")

print(f"The best C value is: {best_C} with an accuracy of: {accuracy_results[best_C]}")


Accuracy results for each C value:
C=0.01: Accuracy=0.898
C=0.1: Accuracy=0.9
C=1: Accuracy=0.901
C=10: Accuracy=0.9
C=100: Accuracy=0.901
The best C value is: 1 with an accuracy of: 0.901
